# Day 061 — Solution: Monitoring & Logging

In [ ]:
_MONITORED_API_SRC = '"""monitored_api.py — Day 061: monitoring and logging for AI apps.\n\nRun:  uvicorn monitored_api:app --reload\nDocs: http://localhost:8000/docs\n"""\nimport json\nimport logging\nimport os\nimport time\nfrom datetime import datetime\n\nimport ollama\nfrom fastapi import FastAPI, Request\nfrom pydantic import BaseModel, Field\n\nMODEL   = os.environ.get("MODEL", "llama3.2")\nAPP_VER = "1.0.0"\nLOG_LVL = os.environ.get("LOG_LEVEL", "INFO")\n\n\nclass JsonFormatter(logging.Formatter):\n    """Emit one JSON object per log record."""\n\n    def format(self, record: logging.LogRecord) -> str:\n        entry = {\n            "timestamp": datetime.fromtimestamp(record.created).isoformat(),\n            "level":     record.levelname,\n            "logger":    record.name,\n            "message":   record.getMessage(),\n        }\n        if record.exc_info:\n            entry["exc"] = self.formatException(record.exc_info)\n        return json.dumps(entry)\n\n\ndef setup_logger(name: str, level: str = "INFO") -> logging.Logger:\n    """Return a named logger with a JsonFormatter on stdout."""\n    logger = logging.getLogger(name)\n    logger.setLevel(getattr(logging, level.upper()))\n    if not logger.handlers:\n        h = logging.StreamHandler()\n        h.setFormatter(JsonFormatter())\n        logger.addHandler(h)\n    logger.propagate = False\n    return logger\n\n\nclass MetricsCollector:\n    """Thread-safe request metrics accumulator."""\n\n    def __init__(self):\n        self._requests = 0\n        self._errors   = 0\n        self._latencies: list[float] = []\n\n    def record(self, status_code: int, duration_ms: float) -> None:\n        self._requests += 1\n        if status_code >= 400:\n            self._errors += 1\n        self._latencies.append(duration_ms)\n\n    def summary(self) -> dict:\n        avg        = sum(self._latencies) / len(self._latencies) if self._latencies else 0.0\n        error_rate = self._errors / self._requests if self._requests else 0.0\n        return {\n            "requests":       self._requests,\n            "errors":         self._errors,\n            "avg_latency_ms": round(avg, 1),\n            "error_rate":     round(error_rate, 3),\n        }\n\n    def reset(self) -> None:\n        self._requests = 0\n        self._errors   = 0\n        self._latencies.clear()\n\n\nclass AskRequest(BaseModel):\n    prompt: str = Field(min_length=1)\n\n\ndef build_api(process_fn=None) -> FastAPI:\n    """Build the monitored API.\n\n    process_fn: optional callable(prompt: str) -> str for testing.\n    """\n    app       = FastAPI(title="Monitored API", version=APP_VER)\n    collector = MetricsCollector()\n    logger    = setup_logger("monitored_api", LOG_LVL)\n\n    @app.middleware("http")\n    async def metrics_middleware(request: Request, call_next):\n        start    = time.monotonic()\n        response = await call_next(request)\n        duration = (time.monotonic() - start) * 1000\n        collector.record(response.status_code, duration)\n        entry = {"method": request.method, "path": request.url.path,\n                 "status": response.status_code, "duration_ms": round(duration, 1)}\n        logger.info(json.dumps(entry))\n        return response\n\n    @app.get("/health")\n    def health():\n        return {"status": "ok",\n                "timestamp": datetime.utcnow().isoformat() + "Z",\n                "version": APP_VER}\n\n    @app.get("/metrics")\n    def metrics():\n        return collector.summary()\n\n    @app.post("/ask")\n    def ask(req: AskRequest):\n        if process_fn is not None:\n            answer = process_fn(req.prompt)\n        else:\n            resp   = ollama.chat(\n                model=MODEL,\n                messages=[{"role": "user", "content": req.prompt}],\n            )\n            answer = resp["message"]["content"]\n        return {"answer": answer}\n\n    return app\n\n\napp = build_api()\n\nif __name__ == "__main__":\n    import uvicorn\n    PORT = int(os.environ.get("PORT", "8000"))\n    uvicorn.run(app, host="0.0.0.0", port=PORT)\n'
from pathlib import Path
Path('monitored_api.py').write_text(_MONITORED_API_SRC)
print('monitored_api.py written.')

In [ ]:
# inline test — no Ollama needed
import json, logging, time
from datetime import datetime
from fastapi import FastAPI, Request
from pydantic import BaseModel, Field
from starlette.testclient import TestClient

# --- JsonFormatter ---
class JsonFormatter(logging.Formatter):
    def format(self, record):
        entry = {
            "timestamp": datetime.fromtimestamp(record.created).isoformat(),
            "level":     record.levelname,
            "logger":    record.name,
            "message":   record.getMessage(),
        }
        if record.exc_info:
            entry["exc"] = self.formatException(record.exc_info)
        return json.dumps(entry)

# --- MetricsCollector ---
class MetricsCollector:
    def __init__(self):
        self._requests = 0; self._errors = 0; self._latencies = []
    def record(self, status_code, duration_ms):
        self._requests += 1
        if status_code >= 400: self._errors += 1
        self._latencies.append(duration_ms)
    def summary(self):
        avg = sum(self._latencies) / len(self._latencies) if self._latencies else 0.0
        rate = self._errors / self._requests if self._requests else 0.0
        return {"requests": self._requests, "errors": self._errors,
                "avg_latency_ms": round(avg, 1), "error_rate": round(rate, 3)}
    def reset(self):
        self._requests = 0; self._errors = 0; self._latencies.clear()

# --- build_api ---
def build_api(process_fn=None):
    app = FastAPI(); collector = MetricsCollector()

    @app.middleware("http")
    async def _m(request: Request, call_next):
        start = time.monotonic()
        response = await call_next(request)
        collector.record(response.status_code, (time.monotonic() - start) * 1000)
        return response

    class _AskReq(BaseModel):
        prompt: str = Field(min_length=1)

    @app.get("/health")
    def health():
        return {"status": "ok", "version": "1.0.0"}

    @app.get("/metrics")
    def metrics():
        return collector.summary()

    @app.post("/ask")
    def ask(req: _AskReq):
        answer = process_fn(req.prompt) if process_fn else req.prompt.upper()
        return {"answer": answer}

    return app

# --- tests ---
app    = build_api(process_fn=lambda p: f"Answer: {p}")
client = TestClient(app, raise_server_exceptions=False)

r = client.get("/health")
assert r.status_code == 200 and r.json()["status"] == "ok"
print("\u2705 /health works")

r2 = client.post("/ask", json={"prompt": "hello"})
assert r2.status_code == 200 and "Answer" in r2.json()["answer"]
print("\u2705 POST /ask returns answer")

r3 = client.post("/ask", json={"prompt": ""})
assert r3.status_code == 422
print("\u2705 empty prompt \u2192 422")

rm = client.get("/metrics")
m = rm.json()
assert m["requests"] == 3
assert m["errors"] == 1
assert m["avg_latency_ms"] >= 0
print("\u2705 /metrics reports correct counts")

# JsonFormatter check
fmt = JsonFormatter()
rec = logging.LogRecord(name="test", level=logging.INFO, pathname="",
                        lineno=0, msg="solution check", args=(), exc_info=None)
data = json.loads(fmt.format(rec))
assert data["level"] == "INFO" and data["message"] == "solution check"
print("\u2705 JsonFormatter produces valid JSON")

# make_health_report check (inline)
def make_health_report(services):
    results = {}
    for name, fn in services.items():
        try: results[name] = bool(fn())
        except Exception: results[name] = False
    if not results: status = "ok"
    elif all(results.values()): status = "ok"
    elif not any(results.values()): status = "down"
    else: status = "degraded"
    return {"status": status, "services": results}

assert make_health_report({"db": lambda: True})["status"] == "ok"
assert make_health_report({"db": lambda: False})["status"] == "down"
assert make_health_report({"db": lambda: True, "cache": lambda: False})["status"] == "degraded"
print("\u2705 make_health_report status logic correct")

print("\nDay 061 \u2014 Monitoring & Logging complete! \U0001f389")
